# 15 - Asyncio


## 1. Getting Started with `asyncio` (Coroutines & Event Loop)

`asyncio`, tek bir iş parçacığı (single-threaded) üzerinde olay döngüsü (event loop) aracılığıyla eşzamanlı (concurrent) G/Ç (I/O) işlemleri yürütmek için kullanılan standart Python kütüphanesidir[cite: 73].

**Syntax**

```python
import asyncio

async def main() -> None:
    await some_coroutine()

asyncio.run(main())
```

**Temel Kavramlar ve Dikkat Edilmesi Gerekenler**
* **Coroutine (`async def`):** Çalışması duraklatılabilen (suspend) ve kaldığı yerden devam ettirilebilen özel fonksiyonlardır[cite: 73]. Çağrıldığında doğrudan çalışmaz, bir coroutine nesnesi döndürür.
* **`await`:** Mevcut coroutine'in yürütmesini ilgili işlem tamamlanana kadar askıya alır ve bu bekleme süresinde olay döngüsünün (event loop) diğer hazır görevleri çalıştırmasına izin verir[cite: 73].
* **`asyncio.run()`:** Olay döngüsünü başlatan, giriş noktası (entry point) coroutine'ini çalıştıran ve tamamlandığında döngüyü kapatan ana çağrıdır[cite: 73]. *(Not: Jupyter/IPython ortamlarında halihazırda çalışan bir event loop bulunduğu için hücre içinde doğrudan üst düzey `await main()` kullanılır.)*
* **Sıralı `await` Tuzağı:** İki coroutine'i arka arkaya doğrudan `await cor1(); await cor2()` şeklinde çağırmak eşzamanlılık sağlamaz; ikincisi birincisinin bitmesini bekler[cite: 73]. Eşzamanlı yürütme için `asyncio.create_task()`, `asyncio.gather()` veya `asyncio.TaskGroup` kullanılmalıdır[cite: 71, 73].


In [ ]:
import asyncio


async def fetch_record(record_id: int, delay: float) -> dict[str, int | str]:
    print(f'Fetching record {record_id}...')
    await asyncio.sleep(delay)
    return {'id': record_id, 'status': 'ok'}


async def count_ticks(limit: int, interval: float) -> None:
    for tick in range(1, limit + 1):
        await asyncio.sleep(interval)
        print(f'Tick {tick}')


async def main() -> None:
    fetch_task: asyncio.Task[dict[str, int | str]] = asyncio.create_task(fetch_record(101, 1.2))
    tick_task: asyncio.Task[None] = asyncio.create_task(count_ticks(4, 0.4))

    record: dict[str, int | str] = await fetch_task
    print(f'Received: {record}')
    await tick_task


await main()


## 2. Tasks, Cancellation & Timeouts (`create_task`, `wait_for`, `timeout`)

`asyncio.Task`, bir coroutine'i sarmalayarak (wrap) olay döngüsünde en kısa sürede çalıştırılmak üzere kuyruğa alan ve gelecekte tamamlanacak bir sonucu temsil eden `Future` alt sınıfıdır[cite: 69].

**Syntax**

```python
task = asyncio.create_task(coro, name='worker')
task.cancel()
task.done()
task.cancelled()
result = task.result()

await asyncio.wait_for(task, timeout=5.0)

async with asyncio.timeout(5.0):
    await task
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**

* **İptal (`cancel` & `CancelledError`):** `task.cancel()` çağrısı görevi bir sonraki `await` noktasında iptal edilmek üzere işaretler[cite: 69]. İptal edilen görev `await` edildiğinde `asyncio.CancelledError` fırlatır[cite: 69]. `task.cancelled()` çağrısının `True` dönmesi için olay döngüsüne iptali işleyecek en az bir geçiş fırsatı (`await asyncio.sleep(0)`) verilmelidir[cite: 69, 72].
* **Durum Kontrolü (`done` vs `result`):** Görev tamamlanmadan `task.result()` çağrılırsa `asyncio.InvalidStateError` fırlar; öncesinde `task.done()` kontrol edilmeli veya `await task` kullanılmalıdır[cite: 69].
* **Zaman Aşımı (`wait_for` & `asyncio.timeout`):** `asyncio.wait_for()` belirtilen sürede tamamlanmayan görevi otomatik iptal eder ve `TimeoutError` (`asyncio.TimeoutError`) fırlatır[cite: 69]. Python 3.11+ ile gelen `async with asyncio.timeout(seconds):` bağlam yöneticisi birden fazla `await` işlemini tek bir zaman bütçesi altında yönetmek için daha temiz bir alternatiftir.


In [ ]:
import asyncio


async def slow_query(duration: float) -> dict[str, str]:
    await asyncio.sleep(duration)
    return {'result': 'completed'}


async def main() -> None:
    task: asyncio.Task[dict[str, str]] = asyncio.create_task(slow_query(5.0))
    await asyncio.sleep(0.2)
    print(f'Done before cancel: {task.done()}')

    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print(f'Task cancelled: {task.cancelled()}')

    try:
        async with asyncio.timeout(0.3):
            await slow_query(2.0)
    except TimeoutError:
        print('Operation timed out via asyncio.timeout')


await main()


## 3. Cooperative Multitasking & Event Loop Starvation

`asyncio` işbirlikçi çoklu görev (cooperative multitasking) modelini kullanır; işletim sistemi iş parçacıkları gibi görevleri zorla kesmez (preemptive değildir)[cite: 70]. Bir coroutine kontrolü yalnızca bir `await` ifadesine ulaştığında olay döngüsüne geri verir[cite: 70].

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Olay Döngüsü Kilitlenmesi (Starvation):** Uzun süren senkron bir döngü veya yoğun CPU hesaplaması içinde hiç `await` noktası yoksa, olay döngüsü diğer bekleyen görevleri başlatamaz ve `task.cancel()` sinyallerini dahi işleyemez[cite: 70].
* **Kontrol Noktaları (Checkpoints):** Uzun döngülerde belirli aralıklarla `await asyncio.sleep(0)` eklemek, ekstra bekleme süresi eklemeden kontrolü olay döngüsüne devreder ve diğer coroutine'lerin/iptal sinyallerinin çalışmasına olanak tanır[cite: 70, 72].
* **CPU-Bound İşler:** Gerçek CPU yoğun işlemler için `asyncio.to_thread()` veya `ProcessPoolExecutor` kullanılmalıdır.


In [ ]:
import asyncio


async def cooperative_counter(limit: int) -> int:
    total: int = 0
    for i in range(1, limit + 1):
        total += i
        if i % 50_000 == 0:
            await asyncio.sleep(0)
    return total


async def main() -> None:
    worker: asyncio.Task[int] = asyncio.create_task(cooperative_counter(50_000_000))
    await asyncio.sleep(0.05)
    worker.cancel()

    try:
        await worker
    except asyncio.CancelledError:
        print('Cooperative worker responded to cancellation immediately.')


await main()


## 4. Running Multiple Coroutines (`asyncio.gather` & `TaskGroup`)

Birden fazla coroutine'i eşzamanlı olarak başlatıp hepsinin tamamlanmasını beklemek ve sonuçlarını toplamak için `asyncio.gather()` veya Python 3.11+ ile gelen yapısal eşzamanlılık (structured concurrency) aracı `asyncio.TaskGroup` kullanılır[cite: 71].

**Syntax**
```python
results = await asyncio.gather(coro1(), coro2(), return_exceptions=True)

async with asyncio.TaskGroup() as tg:
    t1 = tg.create_task(coro1())
    t2 = tg.create_task(coro2())
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Sonuç Sırası:** `asyncio.gather()`, görevler farklı sürelerde tamamlansa bile sonuçları girdide verilen coroutine sırasıyla bir liste olarak döndürür[cite: 71].
* **Hata Yönetimi (`return_exceptions`):** Varsayılan `return_exceptions=False` ayarında bir görev hata fırlatırsa `gather()` hatayı anında yükseltir[cite: 71]. `return_exceptions=True` verildiğinde ise hatalar program akışını kesmez ve sonuç listesinin ilgili indeksine istisna nesnesi olarak yerleştirilir[cite: 71].
* **`asyncio.TaskGroup` (Python 3.11+):** Görevlerden biri hata verdiğinde gruptaki diğer aktif görevleri otomatik olarak iptal eder ve tüm hataları `ExceptionGroup` (`except*`) altında toplar; yetim (orphan) görev kalmasını önlediği için modern Python'da önerilen yaklaşımdır.


In [ ]:
import asyncio


async def load_item(item_id: int) -> dict[str, int]:
    await asyncio.sleep(0.2)
    if item_id == 0:
        raise ValueError('Invalid item ID: 0')
    return {'item_id': item_id}


async def main() -> None:
    gathered: list[dict[str, int] | BaseException] = await asyncio.gather(
        load_item(1),
        load_item(2),
        load_item(0),
        return_exceptions=True,
    )
    print('Gather results:', gathered)

    async with asyncio.TaskGroup() as tg:
        t1: asyncio.Task[dict[str, int]] = tg.create_task(load_item(10))
        t2: asyncio.Task[dict[str, int]] = tg.create_task(load_item(20))

    print('TaskGroup results:', [t1.result(), t2.result()])


await main()


## 5. `asyncio.sleep()` Details (Zero-Delay Yielding & `result`)

`asyncio.sleep()`, mevcut coroutine'i belirtilen saniye boyunca engellemeden (non-blocking) askıya alır ve süresi dolduğunda isteğe bağlı olarak bir dönüş değeri sağlar[cite: 72].

**Syntax**
```python
value = await asyncio.sleep(delay, result=None)
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Sıfır Gecikmeli Kontrol Devri (`delay=0`):** `await asyncio.sleep(0)` çağrısı gerçek bir zaman gecikmesi eklemeden olay döngüsünde bir tur (tick) devir yaparak diğer bekleyen görevlerin ve iptal kontrollerinin çalışmasına fırsat tanır[cite: 72].
* **`result` Parametresi:** İkinci parametre olan `result`, bekleme süresi tamamlandığında `await` ifadesinin döndüreceği değeri belirler; özellikle testlerde ve prototiplerde ağ/veritabanı gecikmesini taklit ederek sahte (mock) veri döndürmek için kullanışlıdır[cite: 72].


In [ ]:
import asyncio


async def main() -> None:
    mock_payload: dict[str, list[int]] = await asyncio.sleep(0.2, result={'items': [1, 2, 3]})
    print(f'Mock payload: {mock_payload}')

    instant_yield: str = await asyncio.sleep(0, result='yielded_control')
    print(f'Zero-delay result: {instant_yield}')


await main()
